***Análisis de Ventas y Comportamiento de Clientes en E-commerce (Online Retail Dataset)***
---
El proyecto analiza datos transaccionales de una tienda de e-commerce con sede en el Reino Unido, especializada en productos de regalo y sin presencia física. El dataset cubre aproximadamente un año completo de operación, lo que permite evaluar el desempeño comercial, el comportamiento de los clientes y la evolución temporal de las ventas en un entorno real de negocio.

El modelo de operación es mixto, con predominancia B2C y presencia de clientes B2B de bajo volumen, lo que introduce dinámicas diferenciadas de compra y valor por cliente.

***Objetivo del Proyecto:*** Evaluar el desempeño comercial del e-commerce y generar insights accionables que apoyen la toma de decisiones estratégicas relacionadas con crecimiento de ingresos, segmentación de clientes, optimización del portafolio de productos y expansión por mercados geográficos.

***Alcance del Análisis:*** El análisis abarca desde la preparación y validación de los datos hasta la exploración de métricas clave de negocio. Se estudian los ingresos, el comportamiento de compra de los clientes mediante segmentación RFM, el desempeño de productos, la evolución temporal de las ventas y la distribución geográfica del revenue.

***Consideraciones sobre los Datos:*** Antes del análisis, se tienen en cuenta aspectos críticos del dataset:

- Existencia de cancelaciones/devoluciones identificadas por el número de factura.
- Posibles cantidades y precios atípicos que pueden afectar métricas financieras.
- Registros sin identificador de cliente, comunes en escenarios reales de e-commerce.

***Métricas Clave (KPIs):*** Se trabajó con indicadores fundamentales para la gestión comercial, incluyendo revenue total y mensual, número de transacciones, Average Order Value (AOV), clientes únicos, frecuencia y recencia de compra, revenue por cliente y concentración de ventas por producto y cliente.

***Valor del Proyecto:*** Este proyecto demuestra la capacidad de:

- Entender un negocio real de e-commerce desde los datos
- Priorizar métricas relevantes para la toma de decisiones
- Detectar patrones, riesgos y oportunidades
- Traducir análisis en insights accionables

***Impacto Esperado en el Negocio:*** Al finalizar el proyecto, se espera entregar:

- Una visión clara del desempeño del negocio
- Segmentación estratégica de clientes
- Identificación de productos críticos y oportunidades
- Recomendaciones concretas para mejorar revenue, retención y eficiencia

---

### ***Carga de Librerías*** 

Se importan las librerías que se utilizarán durante todo el proyecto para:

- Manipulación y análisis de datos transaccionales
- Operaciones numéricas de soporte
- Visualización de datos
- Manejo de fechas para análisis temporal

In [42]:
# Manipulación y análisis de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Manejo de fechas
from datetime import datetime

### ***Carga del Dataset***

En este paso se realiza la carga del dataset desde la ruta local hacia un DataFrame de pandas.


In [43]:
# Ruta del archivo
file_path = r"C:\Users\DIEGO TASCON\Desktop\Datasets\1. Retail, Consumo Masivo & E-commerce\1. Online Retail Dataset\online_retail.csv"

# Carga del dataset
df = pd.read_csv(file_path)

# Vista inicial para confirmar carga correcta
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [44]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


*El dataset contiene ***541.909 registros y 8 variables*** correspondientes a transacciones de un e-commerce. La mayoría de las columnas están completas; no obstante, ``Description`` presenta un bajo porcentaje de valores nulos (~0,3%) y ``CustomerID`` concentra una proporción relevante de datos faltantes (~25%), lo que condiciona los análisis de clientes y requiere separar métricas de negocio de métricas de cliente. Las variables numéricas ``Quantity`` y ``UnitPrice`` están correctamente tipadas para análisis cuantitativo, mientras que ``InvoiceDate`` se encuentra como object y necesita conversión a formato fecha para habilitar análisis temporal. En términos de volumen, el tamaño del dataset (~33 MB) es manejable y adecuado para análisis exploratorio y cálculo de KPIs.*

### ***Estandarización y optimización de variables***

En este paso se ajustan los tipos de datos y se crean variables auxiliares para asegurar que cada columna represente correctamente su rol en el negocio. Esto permite realizar análisis temporales confiables, calcular KPIs sin distorsiones y reflejar eventos reales como cancelaciones y comportamiento de clientes, estableciendo una base sólida y coherente para todo el análisis posterior.

In [45]:
# Conversión de InvoiceDate a datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')

# Conversión de CustomerID a tipo entero (manteniendo nulos)
df['CustomerID'] = df['CustomerID'].astype('Int64')

# Creación de variable para identificar cancelaciones
df['is_cancelled'] = df['InvoiceNo'].str.startswith('C')

# Conversión de Country a categoría (optimización de memoria)
df['Country'] = df['Country'].astype('category')

# Conversión de StockCode a categoría
df['StockCode'] = df['StockCode'].astype('category')

### ***Evaluación inicial de la calidad de los datos***

Se evalúa la calidad general del dataset para identificar posibles problemas que puedan afectar los análisis y KPIs, como valores nulos y registros duplicados. El objetivo es detectar riesgos desde el inicio y definir qué tratamientos serán necesarios antes de avanzar a métricas y análisis de negocio.

In [46]:
# Revisión de valores nulos por variable
df.isna().sum()

InvoiceNo            0
StockCode            0
Description       1454
Quantity             0
InvoiceDate          0
UnitPrice            0
CustomerID      135080
Country              0
is_cancelled         0
dtype: int64

In [47]:
# Revisión de registros duplicados
df.duplicated().sum()

np.int64(5268)

*La revisión de valores nulos muestra que ``CustomerID`` concentra ***135.080 registros faltantes***, lo que confirma una limitación relevante para análisis de clientes y obliga a diferenciar métricas de negocio de métricas basadas en clientes identificados. ``Description`` ***presenta 1.454 valores nulos***, un impacto menor pero a considerar en análisis de productos. El resto de las variables no presenta valores faltantes. Adicionalmente, se identifican ***5.268 registros duplicados***, lo que representa un riesgo de sobreestimación en métricas como revenue, volumen de ventas y número de órdenes si no se tratan adecuadamente antes del análisis.*

In [48]:
# Eliminación de registros duplicados
df = df.drop_duplicates()

# Verificación posterior
df.duplicated().sum()

np.int64(0)

*Tras la eliminación de registros duplicados, el dataset queda con cero duplicados, lo que asegura que cada transacción se contabiliza una única vez. Este ajuste es clave para garantizar la confiabilidad de métricas financieras y operativas, evitando distorsiones en indicadores como revenue, volumen de ventas y número de órdenes, y estableciendo una base sólida para los análisis posteriores.*

### ***Tratamiento de valores nulos***

Se definen y se aplican criterios diferenciados para el tratamiento de valores nulos, priorizando la validez de las métricas de negocio y la consistencia de los análisis de clientes. Se aborda de forma específica el impacto de los nulos en ``Description`` y ``CustomerID``, evitando decisiones que distorsionen KPIs y documentando claramente las limitaciones analíticas resultantes.

In [49]:
# Tratamiento de valores nulos en Description
# Eliminamos registros sin descripción para análisis de productos
df_products = df.dropna(subset=['Description'])

# Separación de datasets para análisis
# Dataset general de negocio (incluye CustomerID nulo)
df_business = df.copy()

# Dataset para análisis de clientes (solo CustomerID válidos)
df_customers = df.dropna(subset=['CustomerID'])

# Verificación de valores nulos restantes
df_business.isna().sum()

InvoiceNo            0
StockCode            0
Description       1454
Quantity             0
InvoiceDate          0
UnitPrice            0
CustomerID      135037
Country              0
is_cancelled         0
dtype: int64

In [50]:
df_customers.isna().sum()

InvoiceNo       0
StockCode       0
Description     0
Quantity        0
InvoiceDate     0
UnitPrice       0
CustomerID      0
Country         0
is_cancelled    0
dtype: int64

*La separación de datasets confirma un ***tratamiento adecuado de los valores nulos***. El dataset de negocio (``df_business``) conserva los valores faltantes en ``Description`` y ``CustomerID``, permitiendo calcular métricas globales sin perder volumen de transacciones, mientras que el dataset de clientes (``df_customers``) queda completamente libre de nulos, lo que garantiza análisis confiables de comportamiento, segmentación y valor del cliente. Esta decisión preserva la integridad de los KPIs y deja explícitas las limitaciones analíticas de cada enfoque.*

### ***Tratamiento de cancelaciones y devoluciones***

Se separan las transacciones válidas de las cancelaciones y devoluciones, ya que estas últimas no representan ventas reales y pueden distorsionar métricas como revenue, número de órdenes y AOV. Esta distinción permite trabajar con ventas netas y mantener, en paralelo, un conjunto de datos útil para analizar fricción operativa y calidad del proceso comercial.

In [51]:
# Dataset de ventas válidas (excluye cancelaciones)
df_sales = df_business[~df_business['is_cancelled']].copy()

# Dataset de cancelaciones y devoluciones
df_cancellations = df_business[df_business['is_cancelled']].copy()

# Verificación de tamaños
df_business.shape, df_sales.shape, df_cancellations.shape

((536641, 9), (527390, 9), (9251, 9))

*Tras el tratamiento de cancelaciones, el dataset de negocio queda compuesto por ***536.641 registros, de los cuales 527.390 corresponden a ventas válidas y 9.251 a cancelaciones o devoluciones***. Esta separación permite trabajar con métricas financieras basadas en ventas reales, evitando distorsiones en KPIs clave, y al mismo tiempo conserva la información necesaria para analizar fricción operativa y calidad del proceso comercial.*

### ***Creación de la variable de revenue***

Se crea la variable revenue a nivel transaccional como el producto de la cantidad vendida y el precio unitario. Esta variable es fundamental para el análisis, ya que permite calcular KPIs financieros, evaluar el desempeño comercial y analizar ingresos por cliente, producto, tiempo y país, siempre basados en ventas reales.

In [52]:
# Creación de la variable revenue
df_sales['revenue'] = df_sales['Quantity'] * df_sales['UnitPrice']

# Verificación inicial de la nueva variable
df_sales[['Quantity', 'UnitPrice', 'revenue']].head()

,Quantity,UnitPrice,revenue
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34


*Se creó correctamente como el producto de ``Quantity`` y ``UnitPrice``, reflejando el ingreso generado por cada línea de transacción.*

### ***Validación de valores atípicos en ``Quantity`` y ``UnitPrice``***

Se evalúan valores extremos en Quantity y UnitPrice para identificar posibles errores o comportamientos atípicos que puedan distorsionar el revenue y los KPIs agregados. Esta validación permite decidir si se requieren filtros o tratamientos antes de consolidar métricas financieras.

In [53]:
# Estadísticos descriptivos para detectar valores atípicos
df_sales[['Quantity', 'UnitPrice', 'revenue']].describe()

# Revisión de valores mínimos y máximos
df_sales[['Quantity', 'UnitPrice']].agg(['min', 'max'])

,Quantity,UnitPrice
min,-9600,-11062.06
max,80995,13541.33


*Los valores mínimos y máximos de ``Quantity`` y ``UnitPrice`` muestran la presencia de ***valores extremos y no plausibles para ventas reales***, incluyendo cantidades y precios negativos, así como valores máximos excesivamente altos. Esto confirma que el dataset contiene ***devoluciones, cancelaciones y posibles errores operativos***, los cuales, de no tratarse, distorsionarían significativamente métricas clave como revenue, AOV y análisis de productos. Por tanto, es necesario definir ***reglas de filtrado coherentes con el negocio*** antes de avanzar a métricas agregadas y visualizaciones.*

### ***Filtrado de valores inválidos en Quantity y UnitPrice***

Se aplican reglas de negocio para eliminar registros con ``Quantity`` o ``UnitPrice`` ***no plausibles***, incluyendo valores negativos (excluyendo devoluciones ya identificadas) y precios excesivamente altos que probablemente son errores. Esto asegura que los KPIs financieros y de productos reflejen únicamente ventas reales y confiables, estableciendo una base sólida para análisis y visualizaciones posteriores.

In [54]:
# Filtrado de registros con Quantity y UnitPrice válidos
# Manteniendo Quantity positivo y UnitPrice positivo
df_sales_clean = df_sales[(df_sales['Quantity'] > 0) & (df_sales['UnitPrice'] > 0)].copy()

# Verificación de nuevos límites
df_sales_clean[['Quantity', 'UnitPrice', 'revenue']].agg(['min','max'])

,Quantity,UnitPrice,revenue
min,1,0.001,0.001
max,80995,13541.330,168469.600


*Tras el filtrado, los registros de ``Quantity`` y ``UnitPrice`` negativos fueron eliminados, dejando únicamente ventas reales y válidas. Los valores mínimos ahora reflejan transacciones pequeñas (1 unidad a 0,001 GBP), mientras que los máximos muestran los registros más grandes del negocio, sin distorsión por errores o devoluciones. Esta limpieza garantiza que los KPIs financieros y de producto se calculen sobre ventas confiables, estableciendo una base sólida para análisis agregados, segmentación de clientes y generación de insights accionables.*

### ***Análisis exploratorio de ventas y revenue***

Se realiza un análisis descriptivo inicial de las ventas y el revenue, con el objetivo de identificar patrones generales, tendencias y rangos de desempeño. Esto permite comprender la distribución de transacciones, detectar concentraciones de ventas por monto o frecuencia, y sentar las bases para análisis más avanzados de clientes, productos y tiempo.

In [55]:
# Estadísticos descriptivos de revenue y Quantity
df_sales_clean[['Quantity', 'UnitPrice', 'revenue']].describe()

,Quantity,UnitPrice,revenue
count,524878.000000,524878.000000,524878.000000
mean,10.616600,3.922573,20.275399
std,156.280031,36.093028,271.693566
min,1.000000,0.001000,0.001000
25%,1.000000,1.250000,3.900000
50%,4.000000,2.080000,9.920000
75%,11.000000,4.130000,17.700000
max,80995.000000,13541.330000,168469.600000


In [56]:
# Número de transacciones únicas y total de revenue
total_transactions = df_sales_clean['InvoiceNo'].nunique()
total_revenue = df_sales_clean['revenue'].sum()

total_transactions, total_revenue

(19960, np.float64(10642110.804))

*El análisis descriptivo muestra que el dataset limpio contiene ***524.878 transacciones válidas***, con un ***total de revenue de 10.642.110 GBP***. La mayoría de las transacciones son de volumen y precio bajos, reflejado en la mediana de Quantity (4 unidades) y UnitPrice (2,08 GBP), mientras que algunos registros extremos elevan significativamente la media y la desviación estándar, indicando una concentración de pocas transacciones de alto valor. Esto evidencia que, aunque la gran mayoría de ventas son pequeñas, un número reducido de pedidos de alto valor impacta significativamente el revenue, un patrón típico en Retail y E-commerce que será clave para análisis de clientes y productos.*

### ***Análisis de clientes y segmentación inicial***

Se analiza la base de clientes identificados, evaluando el revenue por cliente, frecuencia de compra y concentración de ingresos. El objetivo es detectar los clientes más valiosos y entender cómo se distribuye el negocio entre ellos, sentando las bases para segmentación RFM y decisiones estratégicas de marketing y fidelización.

In [57]:
# Revenue total por cliente
revenue_per_customer = df_sales_clean.groupby('CustomerID')['revenue'].sum().sort_values(ascending=False)

# Número de transacciones por cliente
transactions_per_customer = df_sales_clean.groupby('CustomerID')['InvoiceNo'].nunique()

# Estadísticos descriptivos de revenue por cliente
revenue_per_customer.describe()

count      4338.000000
mean       2048.688081
std        8985.230220
min           3.750000
25%         306.482500
50%         668.570000
75%        1660.597500
max      280206.020000
Name: revenue, dtype: float64

*El análisis revela que ***4.338 clientes identificados*** generaron revenue en el período, con un promedio de ***2.048 GBP por cliente***, aunque la desviación estándar alta ***(~8.985 GBP)*** indica que el ingreso está ***muy concentrado en unos pocos clientes de alto valor***. La mediana de 668 GBP sugiere que la mayoría de clientes generan ingresos moderados, mientras que el cliente top alcanza ***280.206 GBP***, mostrando un patrón típico de Retail y E-commerce: ***pocos clientes clave contribuyen de manera desproporcionada al revenue total***. Esta información es clave para priorizar estrategias de fidelización y marketing segmentado.*

### ***Análisis de productos y revenue por artículo***

Se evalúa la contribución de cada producto al revenue y al volumen de ventas, con el objetivo de identificar productos estrella, productos con alta rotación y productos que aportan poco, información clave para optimización de catálogo, promociones y decisiones estratégicas de inventario.

In [58]:
# Revenue total por producto
revenue_per_product = df_sales_clean.groupby('Description')['revenue'].sum().sort_values(ascending=False)

# Cantidad total vendida por producto
quantity_per_product = df_sales_clean.groupby('Description')['Quantity'].sum().sort_values(ascending=False)

# Top 10 productos por revenue y cantidad
top10_revenue_products = revenue_per_product.head(10)
top10_quantity_products = quantity_per_product.head(10)

top10_revenue_products

Description
DOTCOM POSTAGE                        206248.77
REGENCY CAKESTAND 3 TIER              174156.54
PAPER CRAFT , LITTLE BIRDIE           168469.60
WHITE HANGING HEART T-LIGHT HOLDER    106236.72
PARTY BUNTING                          99445.23
JUMBO BAG RED RETROSPOT                94159.81
MEDIUM CERAMIC TOP STORAGE JAR         81700.92
POSTAGE                                78101.88
Manual                                 77752.82
RABBIT NIGHT LIGHT                     66870.03
Name: revenue, dtype: float64

In [59]:
top10_quantity_products

Description
PAPER CRAFT , LITTLE BIRDIE           80995
MEDIUM CERAMIC TOP STORAGE JAR        78033
WORLD WAR 2 GLIDERS ASSTD DESIGNS     54951
JUMBO BAG RED RETROSPOT               48371
WHITE HANGING HEART T-LIGHT HOLDER    37872
POPCORN HOLDER                        36749
PACK OF 72 RETROSPOT CAKE CASES       36396
ASSORTED COLOUR BIRD ORNAMENT         36362
RABBIT NIGHT LIGHT                    30739
MINI PAINT SET VINTAGE                26633
Name: Quantity, dtype: int64

*El análisis muestra que los productos que generan ***más revenue*** no siempre coinciden con los de ***mayor volumen de ventas***. Por ejemplo, ***DOTCOM POSTAGE*** lidera en ingresos (206.248 GBP) pero no aparece en el top por cantidad, mientras que ***PAPER CRAFT, LITTLE BIRDIE*** lidera en unidades vendidas (80.995) y también está entre los top ingresos (168.469 GBP). Esto indica una ***concentración de ingresos en pocos productos de alto valor***, mientras que otros productos contribuyen principalmente en volumen. Estos insights son clave para ***optimización de catálogo, estrategias de pricing y promociones***, permitiendo enfocar esfuerzos en productos que maximicen tanto ventas como margen.*

### ***Análisis temporal de ventas y revenue***

Se analiza cómo evoluciona el número de transacciones y el revenue a lo largo del tiempo, permitiendo identificar tendencias, estacionalidad y picos de demanda. Esta información es fundamental para planificación de inventario, campañas de marketing y estrategias de pricing en Retail y E-commerce.

In [60]:
# Extraer mes y año de la fecha de la transacción
df_sales_clean['InvoiceMonth'] = df_sales_clean['InvoiceDate'].dt.to_period('M')

# Revenue total por mes
monthly_revenue = df_sales_clean.groupby('InvoiceMonth')['revenue'].sum()

# Número de transacciones por mes
monthly_transactions = df_sales_clean.groupby('InvoiceMonth')['InvoiceNo'].nunique()

monthly_revenue

InvoiceMonth
2010-12     821452.730
2011-01     689811.610
2011-02     522545.560
2011-03     716215.260
2011-04     536968.491
2011-05     769296.610
2011-06     760547.010
2011-07     718076.121
2011-08     757841.380
2011-09    1056435.192
2011-10    1151263.730
2011-11    1503866.780
2011-12     637790.330
Freq: M, Name: revenue, dtype: float64

In [61]:
monthly_transactions

InvoiceMonth
2010-12    1559
2011-01    1086
2011-02    1100
2011-03    1454
2011-04    1246
2011-05    1681
2011-06    1533
2011-07    1475
2011-08    1361
2011-09    1837
2011-10    2040
2011-11    2769
2011-12     819
Freq: M, Name: InvoiceNo, dtype: int64

*El análisis temporal muestra ***clara estacionalidad y variabilidad en ventas y revenue***. Por ejemplo, ***noviembre y octubre de 2011*** presentan los picos más altos de revenue (1.503.867 GBP y 1.151.264 GBP) y número de transacciones (2.769 y 2.040), probablemente asociados a campañas de temporada o eventos comerciales. En contraste, meses como ***febrero y diciembre*** muestran menor actividad, reflejando patrones típicos de Retail y E-commerce donde ciertas épocas concentran la mayor parte de ingresos. Estos hallazgos permiten planificar inventario, campañas de marketing y estrategias de pricing basadas en demanda estacional.*

### ***Análisis geográfico de clientes y ventas***

Se analiza cómo se distribuyen las ventas y los clientes por país, con el objetivo de identificar mercados clave, concentraciones de revenue y oportunidades de expansión. Este análisis permite priorizar estrategias de marketing, logística y promociones según el aporte geográfico al negocio.

In [62]:
# Revenue total por país
revenue_per_country = df_sales_clean.groupby('Country')['revenue'].sum().sort_values(ascending=False)

# Número de clientes únicos por país
customers_per_country = df_sales_clean.groupby('Country')['CustomerID'].nunique().sort_values(ascending=False)

# Top 10 países por revenue y clientes
top10_revenue_countries = revenue_per_country.head(10)
top10_customers_countries = customers_per_country.head(10)

top10_revenue_countries

C:\Users\DIEGO TASCON\AppData\Local\Temp\ipykernel_18344\3045404902.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  revenue_per_country = df_sales_clean.groupby('Country')['revenue'].sum().sort_values(ascending=False)
C:\Users\DIEGO TASCON\AppData\Local\Temp\ipykernel_18344\3045404902.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  customers_per_country = df_sales_clean.groupby('Country')['CustomerID'].nunique().sort_values(ascending=False)


Country
United Kingdom    9001744.094
Netherlands        285446.340
EIRE               283140.520
Germany            228678.400
France             209625.370
Australia          138453.810
Spain               61558.560
Switzerland         57067.600
Belgium             41196.340
Sweden              38367.830
Name: revenue, dtype: float64

In [63]:
top10_customers_countries

Country
United Kingdom    3920
Germany             94
France              87
Spain               30
Belgium             25
Switzerland         21
Portugal            19
Italy               14
Finland             12
Austria             11
Name: CustomerID, dtype: int64

*El análisis geográfico revela que el ***mercado principal es el Reino Unido***, concentrando ***9.001.744 GBP de revenue y 3.920 clientes***, lo que representa la gran mayoría de las ventas y la base de clientes. Otros ***países como Países Bajos, EIRE y Alemania*** aportan ingresos significativamente menores, mientras que la cantidad de clientes en estos mercados también es reducida. Este patrón indica una ***fuerte concentración geográfica***, lo que sugiere que las estrategias de marketing, logística y expansión deberían priorizar al Reino Unido, pero también considerar oportunidades de crecimiento en los mercados secundarios.*

### ***Síntesis de métricas clave y preparación de KPIs***

Se consolidan los indicadores más relevantes del negocio, combinando ventas, clientes, productos y geografía. El objetivo es resumir el desempeño del e-commerce en métricas claras y accionables, facilitando la comunicación de insights estratégicos y destacando los hallazgos más relevantes.

In [64]:
# Métricas clave generales
total_revenue = df_sales_clean['revenue'].sum()
total_customers = df_sales_clean['CustomerID'].nunique()
total_transactions = df_sales_clean['InvoiceNo'].nunique()
average_order_value = total_revenue / total_transactions
average_revenue_per_customer = total_revenue / total_customers

# Top producto por revenue
top_product_revenue = df_sales_clean.groupby('Description')['revenue'].sum().idxmax()

# Top país por revenue
top_country_revenue = df_sales_clean.groupby('Country')['revenue'].sum().idxmax()

# Síntesis de KPIs
kpis_summary = {
    'Total Revenue (GBP)': total_revenue,
    'Total Customers': total_customers,
    'Total Transactions': total_transactions,
    'Average Order Value (GBP)': average_order_value,
    'Average Revenue per Customer (GBP)': average_revenue_per_customer,
    'Top Product by Revenue': top_product_revenue,
    'Top Country by Revenue': top_country_revenue
}

kpis_summary

C:\Users\DIEGO TASCON\AppData\Local\Temp\ipykernel_18344\1511079371.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  top_country_revenue = df_sales_clean.groupby('Country')['revenue'].sum().idxmax()


{'Total Revenue (GBP)': np.float64(10642110.804),
 'Total Customers': 4338,
 'Total Transactions': 19960,
 'Average Order Value (GBP)': np.float64(533.1718839679359),
 'Average Revenue per Customer (GBP)': np.float64(2453.229784232365),
 'Top Product by Revenue': 'DOTCOM POSTAGE',
 'Top Country by Revenue': 'United Kingdom'}

*El análisis consolidado muestra que el e-commerce generó un ***revenue total de 10.642.110 GBP*** a partir de ***19.960 transacciones*** realizadas por ***4.338 clientes***. El ***Average Order Value (AOV)*** es de ***533 GBP***, mientras que el ***Average Revenue per Customer asciende a 2.453 GBP***, reflejando la concentración de ingresos en un segmento reducido de clientes de alto valor. El producto que más contribuye al revenue es ***DOTCOM POSTAGE***, y el ***Reino Unido*** representa el mercado más importante, tanto en clientes como en ingresos. Estos KPIs sintetizan de manera clara el desempeño del negocio y destacan los insights estratégicos más relevantes para toma de decisiones en Retail y E-commerce.*

### ***Análisis de concentración de valor (Regla 80/20)***

Este análisis evalúa qué tan concentrado está el revenue en clientes y productos, identificando si una proporción reducida de ellos genera la mayor parte de los ingresos. El objetivo es traducir los datos en decisiones accionables, como priorización de clientes clave, optimización de catálogo y enfoque comercial, un enfoque altamente valorado en Retail y E-commerce.

In [65]:
# Revenue por cliente ordenado
customer_revenue = (
    df_sales_clean
    .groupby('CustomerID')['revenue']
    .sum()
    .sort_values(ascending=False)
)

# Revenue acumulado
customer_revenue_cum = customer_revenue.cumsum()
customer_revenue_pct = customer_revenue_cum / customer_revenue.sum()

# Porcentaje de clientes que generan el 80% del revenue
customers_80pct = customer_revenue_pct[customer_revenue_pct <= 0.8]
len(customers_80pct), len(customer_revenue)

(1129, 4338)

In [66]:
# Revenue por producto ordenado
product_revenue = (
    df_sales_clean
    .groupby('Description')['revenue']
    .sum()
    .sort_values(ascending=False)
)

# Revenue acumulado
product_revenue_cum = product_revenue.cumsum()
product_revenue_pct = product_revenue_cum / product_revenue.sum()

# Porcentaje de productos que generan el 80% del revenue
products_80pct = product_revenue_pct[product_revenue_pct <= 0.8]
len(products_80pct), len(product_revenue)

(827, 4026)

*El análisis de concentración confirma una ***fuerte distribución desigual del revenue***, típica en negocios de Retail y E-commerce. Aproximadamente ***26% de los clientes (1.129 de 4.338)*** generan el ***80% del revenue***, lo que indica que el negocio depende de un ***segmento reducido de clientes de alto valor***, clave para estrategias de fidelización y retención. De forma similar, solo ***21% de los productos (827 de 4.026)*** concentran el ***80% de los ingresos***, evidenciando un catálogo donde pocos artículos sostienen la mayor parte del negocio. Estos hallazgos son fundamentales para* ***priorizar clientes estratégicos, optimizar el portafolio de productos y enfocar decisiones comerciales hacia el máximo impacto en ingresos***.

### ***Segmentación de clientes (RFM)***

En este paso se segmentan los clientes a partir de su comportamiento de compra, considerando qué tan reciente fue su última compra (Recency), con qué frecuencia compran (Frequency) y cuánto revenue generan (Monetary). El objetivo es identificar clientes de alto valor, clientes recurrentes, clientes ocasionales y clientes en riesgo, para apoyar decisiones de retención, fidelización y crecimiento del negocio.

In [ ]:
# Construcción de la tabla RFM
# Fecha de referencia para Recency
reference_date = df_sales_clean['InvoiceDate'].max() + pd.Timedelta(days=1)

# Cálculo de métricas RFM
rfm = (
    df_sales_clean
    .groupby('CustomerID')
    .agg({
        'InvoiceDate': lambda x: (reference_date - x.max()).days,  # Recency
        'InvoiceNo': 'nunique',                                     # Frequency
        'revenue': 'sum'                                            # Monetary
    })
    .reset_index()
)

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

In [76]:
# Scoring RFM (criterio robusto y de negocio)
# Recency: cuantiles (más reciente = mejor score)
rfm['R_Score'] = pd.qcut(
    rfm['Recency'],
    q=4,
    labels=[4, 3, 2, 1]
)

# Monetary: cuantiles (mayor gasto = mejor score)
rfm['M_Score'] = pd.qcut(
    rfm['Monetary'],
    q=4,
    labels=[1, 2, 3, 4]
)

# Frequency: reglas de negocio (más realista para e-commerce)
rfm['F_Score'] = pd.cut(
    rfm['Frequency'],
    bins=[0, 1, 3, 10, rfm['Frequency'].max()],
    labels=[1, 2, 3, 4]
)

In [80]:
# Convertimos scores a numéricos
rfm['R_Score'] = rfm['R_Score'].astype(int)
rfm['F_Score'] = rfm['F_Score'].astype(int)
rfm['M_Score'] = rfm['M_Score'].astype(int)

# Score RFM total
rfm['RFM_Total'] = rfm[['R_Score', 'F_Score', 'M_Score']].sum(axis=1)

# Segmentación basada en score total
def rfm_segment(score):
    if score >= 10:
        return 'Alto valor'
    elif score >= 7:
        return 'Valor medio'
    else:
        return 'Bajo valor'

rfm['Segment'] = rfm['RFM_Total'].apply(rfm_segment)

# Clientes por segmento
rfm['Segment'].value_counts()

# Revenue por segmento
rfm.groupby('Segment')['Monetary'].sum().sort_values(ascending=False)

Segment
Alto valor     6330789.990
Valor medio    1772784.292
Bajo valor      783634.612
Name: Monetary, dtype: float64

*La segmentación RFM muestra una ***alta concentración del revenue en clientes de alto valor***, donde el segmento ***“Alto valor” genera aproximadamente 6,33 millones GBP***, representando la mayor parte de los ingresos totales. El segmento de ***“Valor medio” aporta cerca de 1,77 millones GBP***, mientras que los clientes de ***“Bajo valor” contribuyen solo 0,78 millones GBP***, evidenciando un impacto marginal en el desempeño global del negocio. Este patrón confirma que ***una proporción reducida de clientes sostiene el revenue***, lo que refuerza la necesidad de estrategias enfocadas en ***retención y fidelización de clientes de alto valor***, así como planes de activación para clientes de valor medio.*

---

### ***Conclusiones clave del análisis***

***1. Alta concentración del revenue:*** El negocio presenta una fuerte dependencia de un grupo reducido de clientes y productos. Aproximadamente una cuarta parte de los clientes y una quinta parte del catálogo generan la mayoría de los ingresos, lo que confirma un patrón claro de concentración de valor.

***2. Clientes de alto valor como pilar del negocio:*** El segmento de clientes de Alto valor concentra más del 60% del revenue total, mientras que los clientes de Bajo valor tienen un impacto marginal. Esto indica que la rentabilidad del negocio depende principalmente de la retención y fidelización, más que de la adquisición masiva.

***3. Productos con alto impacto en ingresos:*** Los productos con mayor revenue no siempre son los de mayor volumen, lo que sugiere oportunidades de optimización de pricing, bundles y promociones enfocadas en artículos de alto valor.

***4. Estacionalidad marcada en ventas:*** El revenue y el volumen de transacciones presentan picos claros en determinados meses, lo que permite anticipar la demanda y planificar inventario, campañas de marketing y recursos operativos con mayor precisión.

***5. Fuerte concentración geográfica:*** El Reino Unido domina ampliamente tanto en clientes como en revenue, lo que lo posiciona como el mercado prioritario, mientras que otros países representan oportunidades secundarias de crecimiento controlado.